# TripSense AI - Dataset Exploration

**Purpose:** Initial exploration and validation of the hotel reviews dataset

**Goals:**
1. Load and inspect the raw dataset
2. Validate schema and data types
3. Analyze data quality (missing values, duplicates)
4. Explore distributions and statistics
5. Calculate interaction sparsity
6. Create data dictionary

**Status:** Ready to run after dataset acquisition

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json

# Configure display
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✅ Libraries imported successfully")

## 1. Load Dataset

In [ ]:
# Define data path
DATA_PATH = Path('../data/raw')

# List files in raw directory
print("Files in data/raw/:")
for file in DATA_PATH.glob('*'):
    if not file.name.startswith('.'):
        print(f"  - {file.name} ({file.stat().st_size / 1024 / 1024:.2f} MB)")

In [ ]:
# TODO: Update filename to match your downloaded dataset
DATASET_FILE = 'hotel_reviews.csv'  # Change this to your actual filename

# Load dataset
print(f"Loading {DATASET_FILE}...")
df = pd.read_csv(DATA_PATH / DATASET_FILE)

print(f"\n✅ Dataset loaded successfully")
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")

## 2. Initial Inspection

In [ ]:
# Display first few rows
print("First 5 rows:")
df.head()

In [ ]:
# Display basic info
print("Dataset Info:")
df.info()

In [ ]:
# Display column names
print("Column Names:")
for i, col in enumerate(df.columns, 1):
    print(f"  {i}. {col}")

## 3. Schema Validation

Identify which columns correspond to our expected schema:

In [ ]:
# TODO: Map actual column names to expected names
# Update this mapping based on your dataset's actual columns

column_mapping = {
    # Expected -> Actual
    'user_id': 'user_id',           # UPDATE: e.g., 'User_ID', 'reviewer_id'
    'hotel_id': 'hotel_id',         # UPDATE: e.g., 'Hotel_ID', 'property_id'
    'rating': 'rating',             # UPDATE: e.g., 'Rating', 'overall_rating'
    'review_text': 'review_text',   # UPDATE: e.g., 'Review', 'review_content'
    'hotel_name': 'hotel_name',     # UPDATE: if exists
    'destination': 'destination',   # UPDATE: if exists
    'review_date': 'review_date',   # UPDATE: if exists
}

print("Column Mapping:")
for expected, actual in column_mapping.items():
    exists = actual in df.columns
    status = "✅" if exists else "❌"
    print(f"  {status} {expected} → {actual}")

## 4. Data Quality Analysis

In [ ]:
# Missing values
print("Missing Values:")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Percentage': missing_pct
})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing Count', ascending=False)
print(missing_df)

In [ ]:
# Duplicate records
duplicates = df.duplicated().sum()
print(f"\nDuplicate Records: {duplicates:,} ({duplicates/len(df)*100:.2f}%)")

In [ ]:
# Data types
print("\nData Types:")
print(df.dtypes)

## 5. Core Field Analysis

In [ ]:
# Identify user and hotel columns
# TODO: Update these based on actual column names
user_col = 'user_id'
hotel_col = 'hotel_id'
rating_col = 'rating'
review_col = 'review_text'

print("Dataset Statistics:")
print(f"  Total Records: {len(df):,}")
print(f"  Unique Users: {df[user_col].nunique():,}")
print(f"  Unique Hotels: {df[hotel_col].nunique():,}")
print(f"  Date Range: {df[review_col].min() if review_col in df.columns else 'N/A'} to {df[review_col].max() if review_col in df.columns else 'N/A'}")

In [ ]:
# Rating distribution
print("\nRating Distribution:")
print(df[rating_col].value_counts().sort_index())

plt.figure(figsize=(10, 6))
df[rating_col].value_counts().sort_index().plot(kind='bar')
plt.title('Rating Distribution')
plt.xlabel('Rating')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
# Rating statistics
print("\nRating Statistics:")
print(df[rating_col].describe())

## 6. Interaction Sparsity Analysis

In [ ]:
# Calculate sparsity
n_users = df[user_col].nunique()
n_hotels = df[hotel_col].nunique()
n_interactions = len(df)
possible_interactions = n_users * n_hotels
sparsity = 1 - (n_interactions / possible_interactions)

print("\nInteraction Matrix Statistics:")
print(f"  Users: {n_users:,}")
print(f"  Hotels: {n_hotels:,}")
print(f"  Actual Interactions: {n_interactions:,}")
print(f"  Possible Interactions: {possible_interactions:,}")
print(f"  Sparsity: {sparsity:.4f} ({sparsity*100:.2f}%)")

In [ ]:
# User activity distribution
user_activity = df.groupby(user_col).size()

print("\nUser Activity Statistics:")
print(user_activity.describe())

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
user_activity.hist(bins=50)
plt.title('User Activity Distribution')
plt.xlabel('Number of Reviews per User')
plt.ylabel('Frequency')

plt.subplot(1, 2, 2)
user_activity[user_activity <= 20].hist(bins=20)
plt.title('User Activity Distribution (≤20 reviews)')
plt.xlabel('Number of Reviews per User')
plt.ylabel('Frequency')

plt.tight_layout()
plt.show()

In [ ]:
# Hotel popularity distribution
hotel_popularity = df.groupby(hotel_col).size()

print("\nHotel Popularity Statistics:")
print(hotel_popularity.describe())

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
hotel_popularity.hist(bins=50)
plt.title('Hotel Popularity Distribution')
plt.xlabel('Number of Reviews per Hotel')
plt.ylabel('Frequency')

plt.subplot(1, 2, 2)
hotel_popularity[hotel_popularity <= 100].hist(bins=20)
plt.title('Hotel Popularity Distribution (≤100 reviews)')
plt.xlabel('Number of Reviews per Hotel')
plt.ylabel('Frequency')

plt.tight_layout()
plt.show()

## 7. Review Text Analysis

In [ ]:
# Review text statistics
if review_col in df.columns:
    df['review_length'] = df[review_col].fillna('').str.len()
    df['review_word_count'] = df[review_col].fillna('').str.split().str.len()
    
    print("\nReview Text Statistics:")
    print("\nCharacter Length:")
    print(df['review_length'].describe())
    
    print("\nWord Count:")
    print(df['review_word_count'].describe())
    
    plt.figure(figsize=(12, 5))
    
    plt.subplot(1, 2, 1)
    df['review_length'].hist(bins=50)
    plt.title('Review Length Distribution')
    plt.xlabel('Characters')
    plt.ylabel('Frequency')
    
    plt.subplot(1, 2, 2)
    df['review_word_count'].hist(bins=50)
    plt.title('Review Word Count Distribution')
    plt.xlabel('Words')
    plt.ylabel('Frequency')
    
    plt.tight_layout()
    plt.show()
else:
    print("❌ Review text column not found")

## 8. Aspect Ratings Analysis (if available)

In [ ]:
# Check for aspect rating columns
aspect_columns = [
    'cleanliness_rating',
    'location_rating',
    'rooms_rating',
    'service_rating',
    'value_rating',
    'sleep_quality_rating'
]

# Find matching columns (case-insensitive)
found_aspects = []
for aspect in aspect_columns:
    matches = [col for col in df.columns if aspect.lower() in col.lower()]
    if matches:
        found_aspects.append(matches[0])

if found_aspects:
    print("\n✅ Found Aspect Rating Columns:")
    for col in found_aspects:
        print(f"  - {col}")
        print(f"    Missing: {df[col].isnull().sum()} ({df[col].isnull().sum()/len(df)*100:.2f}%)")
        print(f"    Mean: {df[col].mean():.2f}")
        print(f"    Std: {df[col].std():.2f}")
        print()
    
    # Plot aspect ratings
    if len(found_aspects) > 0:
        fig, axes = plt.subplots(2, 3, figsize=(15, 10))
        axes = axes.flatten()
        
        for i, col in enumerate(found_aspects[:6]):
            df[col].hist(bins=20, ax=axes[i])
            axes[i].set_title(col)
            axes[i].set_xlabel('Rating')
            axes[i].set_ylabel('Frequency')
        
        plt.tight_layout()
        plt.show()
else:
    print("\n⚠️ No aspect rating columns found")
    print("   Will need to derive aspects from review text using NLP")

## 9. Generate Data Dictionary

In [ ]:
# Generate data dictionary
data_dict = {
    'dataset_name': DATASET_FILE,
    'total_records': len(df),
    'total_columns': len(df.columns),
    'unique_users': df[user_col].nunique(),
    'unique_hotels': df[hotel_col].nunique(),
    'sparsity': float(sparsity),
    'columns': {}
}

for col in df.columns:
    data_dict['columns'][col] = {
        'dtype': str(df[col].dtype),
        'missing_count': int(df[col].isnull().sum()),
        'missing_percentage': float((df[col].isnull().sum() / len(df) * 100)),
        'unique_values': int(df[col].nunique()),
    }
    
    # Add statistics for numeric columns
    if df[col].dtype in ['int64', 'float64']:
        data_dict['columns'][col].update({
            'min': float(df[col].min()),
            'max': float(df[col].max()),
            'mean': float(df[col].mean()),
            'std': float(df[col].std()),
        })

# Save data dictionary
with open('../data/data_dictionary.json', 'w') as f:
    json.dump(data_dict, f, indent=2)

print("\n✅ Data dictionary saved to: data/data_dictionary.json")

## 10. Summary & Recommendations

In [ ]:
print("="*80)
print("DATASET VALIDATION SUMMARY")
print("="*80)

print(f"\n✅ Dataset: {DATASET_FILE}")
print(f"✅ Total Records: {len(df):,}")
print(f"✅ Unique Users: {n_users:,}")
print(f"✅ Unique Hotels: {n_hotels:,}")
print(f"✅ Sparsity: {sparsity*100:.2f}%")

print("\n" + "="*80)
print("RECOMMENDATIONS FOR PIPELINE")
print("="*80)

# Collaborative Filtering Viability
if sparsity > 0.99:
    print("\n⚠️  HIGH SPARSITY: Collaborative filtering may be challenging")
    print("   Recommendation: Emphasize content-based and hybrid approaches")
else:
    print("\n✅ Sparsity acceptable for collaborative filtering")

# Aspect Ratings
if found_aspects:
    print(f"\n✅ Aspect ratings available: {len(found_aspects)} aspects found")
else:
    print("\n⚠️  No aspect ratings: Will derive from review text using NLP")

# Data Quality
high_missing = [col for col in df.columns if df[col].isnull().sum() / len(df) > 0.5]
if high_missing:
    print(f"\n⚠️  High missing values in: {', '.join(high_missing)}")
    print("   Recommendation: Consider dropping or imputing these columns")

# Review Text
if review_col in df.columns:
    avg_words = df['review_word_count'].mean()
    print(f"\n✅ Review text available (avg {avg_words:.0f} words)")
    print("   Suitable for NLP-based sentiment analysis")

print("\n" + "="*80)
print("NEXT STEPS")
print("="*80)
print("\n1. Update column_mapping in this notebook with actual names")
print("2. Update .env file with correct column names")
print("3. Document findings in data/DATASET_INFO.md")
print("4. Proceed to Phase 1: Foundation")
print("\n✅ Dataset validation complete!")

## 11. Export Summary for Documentation

In [ ]:
# Create summary report
summary_md = f"""# Dataset Validation Summary

**Date:** {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}
**Dataset:** {DATASET_FILE}

## Overview

- **Total Records:** {len(df):,}
- **Unique Users:** {n_users:,}
- **Unique Hotels:** {n_hotels:,}
- **Sparsity:** {sparsity*100:.2f}%

## Rating Distribution

{df[rating_col].value_counts().sort_index().to_string()}

## Data Quality

### Missing Values

{missing_df.to_string() if len(missing_df) > 0 else 'No missing values'}

### Duplicates

- **Duplicate Records:** {duplicates:,} ({duplicates/len(df)*100:.2f}%)

## Recommendations

{'- ✅ Suitable for collaborative filtering' if sparsity < 0.99 else '- ⚠️ High sparsity - emphasize content-based approaches'}
{'- ✅ Aspect ratings available' if found_aspects else '- ⚠️ Derive aspects from text using NLP'}
- {'✅' if review_col in df.columns else '❌'} Review text available for sentiment analysis

## Next Steps

1. Update configuration with actual column names
2. Proceed to data cleaning pipeline
3. Begin model development
"""

with open('../data/VALIDATION_SUMMARY.md', 'w') as f:
    f.write(summary_md)

print("✅ Validation summary saved to: data/VALIDATION_SUMMARY.md")